# Structure Motif Search Demo

This notebook demonstrates all the 3D structural motif search capabilities provided by the `peptide_motif_search` package. We'll cover:

1. **Motif Definition Format** - JSON structure for defining 3D motifs
2. **Component Definition** - Specifying residues and atom selectors
3. **Constraint Types** - All available geometric constraints
   - Distance constraints
   - Angle constraints
   - Dihedral (torsion) angle constraints
   - Secondary structure constraints
   - Solvent accessibility constraints
   - Exclusion sphere constraints
4. **Running the Search** - Processing PDB/CIF files
5. **Analyzing Results** - Interpreting the output

## Setup


In [ ]:
import sys
import os
import json
import pandas as pd
import numpy as np

# Add the structure_motif directory to the path
sys.path.insert(0, os.path.abspath('../structure_motif'))

# Import the core module
from search_3d_motif import (
    parse_motif_file,
    search_single_file,
    get_atoms_from_structure,
    check_primary_constraints
)

# Define working directories
PROTEIN_FILES_DIR = '../protein_files'
MOTIFS_DIR = '../structure_motif/motifs'
OUTPUT_DIR = '../outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Setup complete!")

---
## 1. Motif Definition Format

Structural motifs are defined using a JSON format with the following structure:

```json
{
  "motif_name": "Name of the motif",
  "description": "Description of what this motif represents",
  "components": [...],   // Residues that make up the motif
  "constraints": [...]   // Geometric relationships between components
}
```

### Available Motif Definitions
Let's explore the pre-defined motif files.

In [ ]:
# View available motif definitions
print("Available Structural Motif Definitions:")
print("="*60)
for f in sorted(os.listdir(MOTIFS_DIR)):
    if f.endswith('.json'):
        filepath = os.path.join(MOTIFS_DIR, f)
        with open(filepath, 'r') as file:
            motif = json.load(file)
        print(f"   Name: {motif['motif_name']}")
        print(f"   Description: {motif['description'][:80]}...")
        print(f"   Components: {len(motif['components'])} residues")
        print(f"   Constraints: {len(motif['constraints'])} rules")
        print(f"\n")

---
## 2. Component Definition

Each component defines a residue in the motif and specifies which atoms to use for geometric calculations.

### Component Fields:
- **id** - Unique identifier for referencing in constraints
- **residue_type** - Three-letter amino acid code (e.g., "SER", "HIS", "ASP")
- **atom_selectors** - Dictionary mapping user-defined names to PDB atom names

In [ ]:
# Examine the catalytic triad motif components
catalytic_triad = parse_motif_file(os.path.join(MOTIFS_DIR, 'catalytic_triad.json'))

print("Catalytic Triad Components:")
print("="*60)
for comp in catalytic_triad['components']:
    print(f"\n Component: {comp['id']}")
    print(f"   Residue Type: {comp['residue_type']}")
    print(f"   Atom Selectors:")
    for name, atom in comp['atom_selectors'].items():
        print(f"      {name}: {atom}")

In [ ]:
# Common PDB atom names for reference
common_atoms = {
    'Backbone': {
        'N': 'Backbone nitrogen',
        'CA': 'Alpha carbon',
        'C': 'Carbonyl carbon',
        'O': 'Carbonyl oxygen'
    },
    'SER (Serine)': {
        'CB': 'Beta carbon',
        'OG': 'Hydroxyl oxygen'
    },
    'HIS (Histidine)': {
        'CB': 'Beta carbon',
        'CG': 'Gamma carbon (imidazole)',
        'ND1': 'Delta nitrogen',
        'CE1': 'Epsilon carbon',
        'NE2': 'Epsilon nitrogen'
    },
    'ASP (Aspartate)': {
        'CB': 'Beta carbon',
        'CG': 'Gamma carbon',
        'OD1': 'Carboxyl oxygen 1',
        'OD2': 'Carboxyl oxygen 2'
    },
    'LYS (Lysine)': {
        'CB': 'Beta carbon',
        'NZ': 'Terminal amine nitrogen'
    },
    'PHE (Phenylalanine)': {
        'CB': 'Beta carbon',
        'CG': 'Gamma carbon (ring)',
        'CZ': 'Zeta carbon (para position)'
    }
}

print("Common PDB Atom Names Reference:")
print("="*60)
for residue, atoms in common_atoms.items():
    print(f"\n{residue}:")
    for atom, desc in atoms.items():
        print(f"   {atom:4s} - {desc}")

---
## 3. Constraint Types

The structural motif search supports six types of constraints:

### 3.1 Distance Constraints
Measure the distance between two atoms in Ångströms.

In [ ]:
# Example distance constraint
distance_constraint = {
    "type": "distance",
    "atoms": ["ser.hydroxyl_oxygen", "his.imidazole_nitrogen_delta"],
    "value": 4.8,      # Target distance in Å
    "tolerance": 0.6   # Allowed deviation in Å
}

print("Distance Constraint Example:")
print(json.dumps(distance_constraint, indent=2))
print(f"\nThis constraint accepts distances from {distance_constraint['value'] - distance_constraint['tolerance']}Å to {distance_constraint['value'] + distance_constraint['tolerance']}Å")

### 3.2 Angle Constraints
Measure the angle formed by three atoms in degrees. The angle is measured at the **middle atom**.

In [ ]:
# Example angle constraint
angle_constraint = {
    "type": "angle",
    "atoms": ["his.imidazole_nitrogen_delta", "ser.hydroxyl_oxygen", "ser.beta_carbon"],
    "value": 89.0,     # Target angle in degrees
    "tolerance": 15.0  # Allowed deviation in degrees
}

print("Angle Constraint Example:")
print(json.dumps(angle_constraint, indent=2))
print(f"\nThe angle is measured at: ser.hydroxyl_oxygen (the middle atom)")
print(f"Acceptable range: {angle_constraint['value'] - angle_constraint['tolerance']}° to {angle_constraint['value'] + angle_constraint['tolerance']}°")


### 3.3 Dihedral (Torsion) Angle Constraints
Measure the dihedral angle formed by four atoms in degrees. This describes the rotation around the bond between atoms 2 and 3.

In [ ]:
# Example dihedral constraint
dihedral_constraint = {
    "type": "dihedral",
    "atoms": [
        "his.alpha_carbon",
        "his.beta_carbon", 
        "his.gamma_carbon",
        "his.imidazole_nitrogen_delta"
    ],
    "value": -109.0,    # Target dihedral in degrees
    "tolerance": 20.0   # Allowed deviation
}

print("Dihedral Constraint Example:")
print(json.dumps(dihedral_constraint, indent=2))
print(f"\nDihedral angle defines the rotation around the CB-CG bond")
print(f"Range: {dihedral_constraint['value'] - dihedral_constraint['tolerance']}° to {dihedral_constraint['value'] + dihedral_constraint['tolerance']}°")

### 3.4 Secondary Structure Constraints
Constrain residues to specific secondary structure types using DSSP assignments.

| Code | Secondary Structure |
|:----:|--------------------|
| H | α-helix |
| B | β-bridge |
| E | Extended strand (β-sheet) |
| G | 3₁₀-helix |
| I | π-helix |
| T | Turn |
| S | Bend |
| - | Other/None |

In [ ]:
# Example secondary structure constraint
ss_constraint = {
    "type": "secondary_structure",
    "component_id": "ser",
    "value": ["T", "S", "-"]  # Accepts Turn, Bend, or Other
}

print("Secondary Structure Constraint Example:")
print(json.dumps(ss_constraint, indent=2))
print(f"\nThis requires the serine to be in a Turn, Bend, or unassigned region")
print("(Not in a helix or β-sheet)")

### 3.5 Accessibility Constraints
Constrain the relative solvent accessibility (RSA) of a residue. Values range from 0.0 (completely buried) to 1.0 (fully exposed).

In [ ]:
# Example accessibility constraint
accessibility_constraint = {
    "type": "accessibility",
    "component_id": "his",
    "comparison": "greater_than",  # or "less_than"
    "value": 0.2  # RSA threshold
}

print("Accessibility Constraint Example:")
print(json.dumps(accessibility_constraint, indent=2))
print(f"\nThis requires the histidine to have RSA > 0.2")
print("(At least 20% of max possible surface exposure - partially exposed)")

print("\nTypical RSA values:")
print("  0.0 - 0.1  : Buried (protein core)")
print("  0.1 - 0.3  : Partially exposed")
print("  0.3 - 0.5  : Exposed")
print("  0.5 - 1.0  : Highly exposed")

### 3.6 Exclusion Sphere Constraints
Define a sphere around an atom that must not contain any other protein atoms. This is useful for ensuring active sites or binding pockets are open and accessible.

In [ ]:
# Example exclusion sphere constraint
exclusion_constraint = {
    "type": "exclusion_sphere",
    "component_id": "ser",
    "atom_selector": "hydroxyl_oxygen",
    "radius": 2.5  # Radius in Å
}

print("Exclusion Sphere Constraint Example:")
print(json.dumps(exclusion_constraint, indent=2))
print(f"\nNo atoms (except from the serine itself) can be within 2.5Å of the hydroxyl oxygen")
print("This ensures the catalytic oxygen is accessible for substrate binding")

---
## 4. Example Motif Definitions

Let's examine the pre-defined motif files in detail.

In [ ]:
# Load and display the basic catalytic triad
with open(os.path.join(MOTIFS_DIR, 'catalytic_triad.json'), 'r') as f:
    basic_triad = json.load(f)

print("Basic Catalytic Triad Motif:")
print("="*60)
print(f"Name: {basic_triad['motif_name']}")
print(f"Description: {basic_triad['description']}")
print(f"\nComponents ({len(basic_triad['components'])}):", [c['id'] for c in basic_triad['components']])
print(f"\nConstraints ({len(basic_triad['constraints'])}) - All Distance:")
for c in basic_triad['constraints']:
    print(f"  {c['atoms'][0]} ↔ {c['atoms'][1]}: {c['value']}±{c['tolerance']}Å")

In [ ]:
# Load and display the advanced catalytic triad (with all constraint types)
with open(os.path.join(MOTIFS_DIR, 'catalytic_triad_advanced.json'), 'r') as f:
    advanced_triad = json.load(f)

print("Advanced Catalytic Triad Motif:")
print("="*60)
print(f"Name: {advanced_triad['motif_name']}")
print(f"\nThis motif uses ALL constraint types:")

constraint_types = {}
for c in advanced_triad['constraints']:
    ctype = c['type']
    if ctype not in constraint_types:
        constraint_types[ctype] = []
    constraint_types[ctype].append(c)

for ctype, constraints in constraint_types.items():
    print(f"\n {ctype.upper()} ({len(constraints)}):")
    for c in constraints:
        if ctype == 'distance':
            print(f"     {c['atoms'][0]} ↔ {c['atoms'][1]}: {c['value']}±{c['tolerance']}Å")
        elif ctype == 'angle':
            print(f"     ({' - '.join(c['atoms'])}): {c['value']}±{c['tolerance']}°")
        elif ctype == 'dihedral':
            print(f"     Torsion({' - '.join(c['atoms'])}): {c['value']}±{c['tolerance']}°")
        elif ctype == 'secondary_structure':
            print(f"     {c['component_id']} must be in: {c['value']}")
        elif ctype == 'accessibility':
            print(f"     {c['component_id']} RSA {c['comparison']} {c['value']}")
        elif ctype == 'exclusion_sphere':
            print(f"     {c['component_id']}.{c['atom_selector']}: radius={c['radius']}Å")

In [ ]:
# Load and display the kinase active site motif
with open(os.path.join(MOTIFS_DIR, 'kinase_active_site.json'), 'r') as f:
    kinase_motif = json.load(f)

print("Kinase Active Site Motif:")
print("="*60)
print(f"Name: {kinase_motif['motif_name']}")
print(f"Description: {kinase_motif['description']}")
print(f"\nComponents:")
for c in kinase_motif['components']:
    print(f"  • {c['id']}: {c['residue_type']}")
print(f"\nThis represents the DFG-in (active) conformation of kinases")

---
## 5. Running the Structure Search

Now let's run a structural motif search on the protein files.

In [ ]:
# View available protein structure files
print("Available Protein Structure Files:")
print("="*60)
structure_files = [f for f in sorted(os.listdir(PROTEIN_FILES_DIR)) 
                   if f.endswith(('.pdb', '.cif'))]
for f in structure_files:
    filepath = os.path.join(PROTEIN_FILES_DIR, f)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  {f:20s} ({size_kb:.0f} KB)")
print(f"\nTotal: {len(structure_files)} structures")

In [ ]:
# Search for catalytic triad in a single structure
motif_def = parse_motif_file(os.path.join(MOTIFS_DIR, 'catalytic_triad.json'))
test_structure = os.path.join(PROTEIN_FILES_DIR, '1AQ7.pdb')  # Known serine protease

print(f"Searching for: {motif_def['motif_name']}")
print(f"In structure: {os.path.basename(test_structure)}")
print("-"*40)

found_motifs = search_single_file(test_structure, motif_def)

if found_motifs:
    print(f"\n Found {len(found_motifs)} motif(s)!")
    for i, motif in enumerate(found_motifs):
        print(f"\nMotif {i+1}:")
        for res in motif['residues']:
            print(f"   {res['res_name']}-{res['chain_id']}-{res['res_id']}")
else:
    print("\n No motifs found")

In [ ]:
# Search across multiple structures
print("Batch Search for Catalytic Triads")
print("="*60)

results_summary = []

# Search subset of structures
for struct_file in structure_files[:10]:  # Limit to first 10 for demo
    filepath = os.path.join(PROTEIN_FILES_DIR, struct_file)
    found = search_single_file(filepath, motif_def)
    
    status = f" {len(found)} found" if found else " None"
    print(f"  {struct_file:20s}: {status}")
    
    results_summary.append({
        'file': struct_file,
        'motifs_found': len(found),
        'matches': found
    })

# Summary
total_found = sum(r['motifs_found'] for r in results_summary)
files_with_motifs = sum(1 for r in results_summary if r['motifs_found'] > 0)
print(f"\nSummary: Found {total_found} motifs in {files_with_motifs}/{len(results_summary)} structures")

---
## 6. Creating Custom Motif Definitions

Let's create a custom structural motif and search for it.

In [ ]:
# Create a custom motif: Simple Cys-Cys disulfide bridge pattern
disulfide_motif = {
    "motif_name": "Potential Disulfide Bridge",
    "description": "Two cysteine residues close enough to form a disulfide bond",
    "components": [
        {
            "id": "cys1",
            "residue_type": "CYS",
            "atom_selectors": {
                "sulfur": "SG",
                "alpha_carbon": "CA"
            }
        },
        {
            "id": "cys2",
            "residue_type": "CYS",
            "atom_selectors": {
                "sulfur": "SG",
                "alpha_carbon": "CA"
            }
        }
    ],
    "constraints": [
        {
            "type": "distance",
            "atoms": ["cys1.sulfur", "cys2.sulfur"],
            "value": 2.05,   # bond length
            "tolerance": 0.3
        }
    ]
}

# Save the custom motif
custom_motif_path = os.path.join(OUTPUT_DIR, 'disulfide_motif.json')
with open(custom_motif_path, 'w') as f:
    json.dump(disulfide_motif, f, indent=2)

print("Custom Disulfide Bridge Motif:")
print(json.dumps(disulfide_motif, indent=2))

In [ ]:
# Search for disulfide bridges
print("Searching for Disulfide Bridges")
print("="*60)

for struct_file in structure_files[:5]:
    filepath = os.path.join(PROTEIN_FILES_DIR, struct_file)
    found = search_single_file(filepath, disulfide_motif)
    
    if found:
        print(f"\n {struct_file}: {len(found)} disulfide bridge(s)")
        for i, match in enumerate(found[:3]):  # Show first 3
            cys1 = match['residues'][0]
            cys2 = match['residues'][1]
            print(f"   Bridge {i+1}: {cys1['res_name']}-{cys1['res_id']} ↔ {cys2['res_name']}-{cys2['res_id']}")
    else:
        print(f" {struct_file}: None found")

In [ ]:
# Create a more complex custom motif: Salt bridge
salt_bridge_motif = {
    "motif_name": "Salt Bridge",
    "description": "Electrostatic interaction between oppositely charged residues",
    "components": [
        {
            "id": "lys",
            "residue_type": "LYS",
            "atom_selectors": {
                "amine_nitrogen": "NZ"
            }
        },
        {
            "id": "asp",
            "residue_type": "ASP",
            "atom_selectors": {
                "carboxyl_oxygen": "OD1"
            }
        }
    ],
    "constraints": [
        {
            "type": "distance",
            "atoms": ["lys.amine_nitrogen", "asp.carboxyl_oxygen"],
            "value": 2.8,   #salt bridge distance
            "tolerance": 0.5
        }
    ]
}

print("Salt Bridge Motif Definition:")
print(json.dumps(salt_bridge_motif, indent=2))

---
## 7. Using the Command Line Interface

The structure search can also be run from the command line:

In [ ]:
# Generate the command line example
print("Command Line Usage:")
print("="*60)
print("""\npython structure_motif/search_3d_motif.py \\
    --input_folder protein_files \\
    --motif_file structure_motif/motifs/catalytic_triad.json \\
    --output_folder outputs \\
    --summary_csv summary.csv""")

print("\nRequired Arguments:")
print("  -i, --input_folder   : Folder containing PDB/CIF files")
print("  -m, --motif_file     : JSON motif definition file")
print("  -o, --output_folder  : Folder to save individual JSON results")
print("  -s, --summary_csv    : Path for the summary CSV file")